In [9]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.datastructures.grasp import GraspDescription
from uniworld import load_pr2_apartment_world
from minimal_llmr.backend import LLMBackend
from dotenv import load_dotenv
from llmr_updated_arch.integrations.langchain.provider import LLMProvider, make_llm
load_dotenv("../.env")

True

In [10]:
exp = Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

In [11]:
world, robot_view, context = load_pr2_apartment_world()

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [12]:
llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))

LLM ready: gpt-4o-mini


In [13]:
backend = LLMBackend(llm=llm, instruction="pick up milk from the table")

In [14]:
action = next(iter(backend.evaluate(exp)))

In [15]:
action.object_designator

Body(name=PrefixedName('None/milk.stl'), id=UUID('aeae14e9-977f-4d63-accc-6ba8c4d86f1b'), index=219)

## LLM Reasoning Inspection

How the LLM resolved each parameter — the value or referent it chose and its justification.

In [16]:
result = backend.last_result
interps = result.interpretations

if interps.overall_reasoning:
    print("Overall reasoning:")
    print(f"  {interps.overall_reasoning}")
    print()

print("Per-parameter reasoning:")
print("─" * 64)
for interp in interps.interpretations:
    print(f"  param    : {interp.param_name}")
    if interp.referent_description:
        rd = interp.referent_description
        print(f"  kind     : referent")
        print(f"  name     : {rd.name}")
        if rd.semantic_type:
            print(f"  type     : {rd.semantic_type}")
        if rd.spatial_context:
            print(f"  loc      : {rd.spatial_context}")
        if rd.attributes:
            print(f"  attrs    : {rd.attributes}")
    else:
        print(f"  kind     : discrete")
        print(f"  value    : {interp.value}")
    print(f"  causal   : {interp.reasoning.causal}")
    print(f"  counter  : {interp.reasoning.counterfactual}")
    print(f"  spatial  : {interp.reasoning.spatial}")
    print(f"  geometry : {interp.reasoning.geometric}")
    print("─" * 64)

if interps.coherence_assessment:
    print("\nCoherence assessment:")
    print(f"  {interps.coherence_assessment}")

Overall reasoning:
  The parameters chosen ensure that the robot can effectively and securely pick up the milk from the table using both arms, approaching from the front and aligning its gripper from the top.

Per-parameter reasoning:
────────────────────────────────────────────────────────────────
  param    : object_designator
  kind     : referent
  name     : milk.stl
  type     : Milk
  loc      : on the table
  causal   : The object to be picked up is specifically identified as 'milk', which matches the semantic type and name.
  counter  : An alternative object could be 'breakfast_cereal.stl', but it is not the target specified in the instruction.
  spatial  : The milk object is located on the table, which is the specified location for the action.
  geometry : The milk object is of a size that is manageable for the robot to grasp.
────────────────────────────────────────────────────────────────
  param    : arm
  kind     : discrete
  value    : BOTH
  causal   : Using both arms 

In [ ]:
# print("Grounded parameters:")
# for name, val in result.grounded_params.items():
#     print(f"  {name}: {val!r}")

In [17]:
# claude --resume fb905fef-518e-4f46-89c2-25202bb2162c